# 第 2 章 · 概率与常见分布

统计推断建立在**概率**之上。本章介绍随机变量与几种最常用的概率分布。

## 0. 准备环境

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 让 matplotlib 正确显示中文与负号
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

np.random.seed(42)   # 固定随机种子，保证每次运行结果一致
print("环境就绪：numpy", np.__version__, "| pandas", pd.__version__)

## 1. 概率的基本概念

- **随机试验**：结果不确定，但所有可能结果已知。
- **样本空间 S**：所有可能结果的集合。
- **事件 A**：样本空间的子集。
- **概率 P(A)**：事件发生的可能性，满足 0 ≤ P(A) ≤ 1，且 P(S)=1。

**加法法则**：P(A∪B) = P(A) + P(B) − P(A∩B)
**独立事件**：P(A∩B) = P(A)·P(B)

In [ ]:
# 抛两枚硬币的样本空间
S = ["正正", "正反", "反正", "反反"]
print("样本空间 S =", S)
# 事件：至少出现一个正面
event = [s for s in S if "正" in s]
print("事件『至少一个正面』=", event, " 概率 =", len(event) / len(S))

## 2. 随机变量与期望

**离散随机变量**：取值可数。期望 E(X)=Σx·P(x)，方差 Var(X)=E[(X−μ)²]。

In [ ]:
# 掷一颗骰子的期望与方差
faces = np.arange(1, 7)
p = np.full(6, 1 / 6)
Ex = np.sum(faces * p)
Varx = np.sum((faces - Ex) ** 2 * p)
print("E(X) =", Ex, " Var(X) = %.4f" % Varx, " 标准差 = %.4f" % math.sqrt(Varx))

## 3. 常见离散分布

**二项分布** B(n, p)：n 次独立试验中成功的次数。
**泊松分布** Poisson(λ)：单位时间/面积内稀有事件发生次数。

In [ ]:
from math import comb

def binom_pmf(k, n, p):
    return comb(n, k) * p ** k * (1 - p) ** (n - k)

n, p = 10, 0.5                      # 抛 10 次硬币
ks = np.arange(0, n + 1)
probs = [binom_pmf(k, n, p) for k in ks]

plt.figure(figsize=(8, 3.5))
plt.bar(ks, probs, color="#6da8f5")
plt.title("二项分布 B(10, 0.5)")
plt.xlabel("成功次数 k"); plt.ylabel("概率")
plt.show()

print("期望 np =", n * p, " 方差 np(1-p) =", n * p * (1 - p))

In [ ]:
def poisson_pmf(k, lam):
    return lam ** k * math.exp(-lam) / math.factorial(k)

lam = 3
ks = np.arange(0, 15)
probs = [poisson_pmf(k, lam) for k in ks]

plt.figure(figsize=(8, 3.5))
plt.bar(ks, probs, color="#f5a06d")
plt.title("泊松分布 Poisson(λ=3)")
plt.xlabel("事件次数 k"); plt.ylabel("概率")
plt.show()

print("期望 = 方差 = λ =", lam)

## 4. 最常用的连续分布：正态分布

记作 X ~ N(μ, σ²)。标准正态 N(0,1) 的分布函数表是统计推断的基础。

**经验法则（68-95-99.7）**：约 68% 落在 μ±σ，95% 落在 μ±2σ，99.7% 落在 μ±3σ。

In [ ]:
def normal_pdf(x, mu, sigma):
    return 1 / (sigma * math.sqrt(2 * math.pi)) * math.exp(-(x - mu) ** 2 / (2 * sigma ** 2))

xs = np.linspace(-8, 8, 400)
plt.figure(figsize=(8, 4))
for mu, sigma, c in [(0, 1, "#2255aa"), (0, 2, "#6da8f5"), (2, 1, "#f5a06d")]:
    plt.plot(xs, [normal_pdf(t, mu, sigma) for t in xs], color=c, label=f"N({mu}, {sigma}²)")
plt.title("不同的正态分布")
plt.legend(); plt.show()

## 5. 标准正态与 Z 分数

**Z 分数**把任意正态分布标准化：Z = (X − μ) / σ ~ N(0, 1)。
Z 表示"这个值离均值有多少个标准差"。

In [ ]:
def z_score(x, mu, sigma):
    return (x - mu) / sigma

# 某次考试成绩均值 75、标准差 8，某同学考了 91 分
z = z_score(91, 75, 8)
print("Z = %.2f，说明该成绩高于均值 %.2f 个标准差" % (z, z))
print("经验法则下约有 %.1f%% 的同学低于他 (Z≈2 -> 约97.7%%)" % 97.7)

# 用 numpy 直接从正态分布抽样验证经验法则
sample = np.random.normal(75, 8, 100000)
for k in (1, 2, 3):
    inside = np.mean(np.abs(sample - 75) <= k * 8)
    print(f"落在 μ±{k}σ 内的比例 = {inside:.4f}")

## 小结

- 概率描述不确定性；随机变量把结果映射为数值。
- 离散：二项、泊松；连续：正态。
- **Z 分数 / 标准正态**是后面置信区间和假设检验的核心工具。